In [2]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, zarr_dir, images_dir, surface_drifters, depth_drifters, depth_100, depth_50
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pynsitu as pyn

In [ ]:
color = {
    'BIOSWOT: CARTHE': "firebrick",
    'FAST-SWOT: CARTHE':'darkred',
    'C-SWOT: CARTHE': "orange",
    'BIOSWOT: CODE': "green",
    'BIOSWOT: SVPOGS': "darkblue",
    'BIOSWOT: SVPSIO': "teal",
    'C-SWOT: SVP': "lightblue",
    'C-SWOT: SPOTTER': "yellow",
    'C-SWOT: MELODI': "hotpink",
    'FAST-SWOT: HEREON':'peru',
    'FAST-SWOT: OSMC':'violet',
}
depthcolor = {0:'navy', 15:'green', 50:'yellowgreen', 100:'yellow'}
depthmarker = {0:'*', 15:'o', 50:'^', 100:'s'}
lw=0.5
bbox = [-0.3, 6.1, 36.5, 43.2]

In [ ]:
# drifters under swot
df = pd.read_csv(os.path.join(zarr_dir, 'coloc_all_nofilter.csv'), parse_dates = ['datetime'])
dfc = pd.read_csv(os.path.join(zarr_dir, 'coloc_nearest_nofilter.csv'), parse_dates = ['datetime'])
df['depthcolor'] = [depthcolor[d] for d in df.depth] 
dfc['depthcolor'] = [depthcolor[d] for d in dfc.depth] 

#SWOT
dfs = browse_swot_250().reset_index()

#ERA5
files = glob('/Users/mdemol/DATA_WIND/era5/*')
era  = xr.open_dataset(files[0])

In [ ]:
df

In [ ]:
df['time'] = (pd.to_datetime(df.datetime) - pd.to_datetime(df.datetime.min()))/pd.Timedelta('1s')

In [ ]:
dfs['time'] = pd.to_datetime(dfs.time)

_______
# One image per SWOT image, with drifters for whose this image is the nearest

In [ ]:
def select_data(cycle, swath, dfs, df, dfc, era):
    # SWOT
    dfs_ = dfs.where(dfs.pass_number==int(swath)).dropna()
    dss = xr.open_dataset(dfs_.set_index('cycle_number').loc[cycle].file)
    dss['eta'] = dss.cvl_mean_dynamic_topography_cnes_cls_22 + dss.cvl_ocean_tide_fes_2022 + dss.duacs_ssha_karin_2_filtered
    dss.eta.attrs={'long_name':'ADT+ocean tide correction', 'unit':'m'}
    dss['adt'] = dss.cvl_mean_dynamic_topography_cnes_cls_22 + dss.cvl_ocean_tide_fes_2022 + dss.duacs_ssha_karin_2_filtered
    dss.eta.attrs={'long_name':'ADT', 'unit':'m'}

    # DRIFTERS
    df_ = df.where((df.pass_number==swath)&(df.cycle_number==cycle)).dropna()
    dfc_ = dfc.where((dfc.pass_number==swath)&(dfc.cycle_number==cycle)).dropna()
    
    #ERA5
    t = dss.time.mean()
    era_ = era.sel(time=dss.time.mean(), method='nearest').sortby('latitude')
    
    return dss, df_, dfc_, era_

def set_legend(handles, labels):
    lab = np.unique(labels)
    ind = [np.argwhere(np.array(labels) == l)[0][0] for l in lab]
    hand = np.take(handles, ind)
    return hand, lab
    #ax.legend(hand, lab)

def plot_one_cycle_one_swath(cycle, swath, dfs, df, dfc, era):
    if swath == 3 : 
        #bbox = [2.7, 6.1, 36.6, 43.4]
        #bbox, pos = [3.5, 6.1, 40, 43.4], 'sup'
        bbox, pos = [2.7, 5.1, 36.6, 40], 'inf'
        
        vmax=0.2
        vmin=-0.2
        
    if swath == 16 : 
        #bbox = [0.9, 3.7, 36.5, 41.5]
        bbox, pos = [0.9, 3.7, 37.7, 41.25], ''
        vmax=0.2
        vmin=-0.2
        
    dss, df_, dfc_, era_ = select_data(cycle, swath, dfs, df, dfc, era)
    
    
    fig = plt.figure(figsize=(10,10))
    fig.patch.set_facecolor('white')#no transparancy in png
    ax = fig.add_subplot(111, projection=ccrs.Orthographic(df_.longitude.mean(), df_.latitude.mean()))
    
    #SWOT
    #(dss.eta - dss.eta.median()).plot(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax,cmap='RdBu_r', cbar_kwargs={'pad':0.1})#"location": "left"})#remove median
    (dss.adt - dss.adt.median()).plot(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax,cmap='RdBu_r', cbar_kwargs={'pad':0.1})#"location": "left"})#remove median
    #dss.eta.plot.contour(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax, levels=40, colors='k', linewidths=0.2)
    dss['ve'] = dss.duacs_speed_zonal_abs 
    dss['vn'] = dss.duacs_speed_meridional_abs
    dss_c = dss[['ve', 'vn']].isel(num_lines = slice(0,None,15),num_pixels = slice(0,None,15))
    dss_c = dss_c.where(dss_c.latitude>bbox[2]).where(dss_c.latitude<bbox[3])
    P = dss_c.plot.quiver('longitude', 'latitude','ve','vn', color='k', scale = 20, transform=crs, ax=ax, clip_on = False, add_guide = False)
    ax.quiverkey(P, 0.8, 0.89, 1, 'SWOT 1/15\n'+r'1 $m.s^{-1}$', coordinates='figure', labelpos='N')

    
    #ERA5
    era_ = era_.sel(longitude=slice(bbox[0], bbox[1]), latitude=slice(bbox[2], bbox[3]))
    Q = era_.plot.quiver('longitude', 'latitude', 'u10', 'v10', ax=ax, transform =crs, scale=250, color='magenta', clip_on = False, add_guide = False)
    ax.quiverkey(Q, 0.8, 0.95, 10, 'ERA5 wd at 10m\n'+r'10 $m.s^{-1}$', coordinates='figure', labelpos='N')
    
    #drifters
    for id_ in df_.drifter_id.unique():
        df__ = df_[df_.drifter_id==id_]
        a = ax.scatter(df__.longitude, df__.latitude, c=df__.time, transform =crs, s = 2, alpha=0.2, cmap='viridis', zorder=10)
        cb = plt.colorbar(a)
        cb.remove()
    #df_.plot.scatter('longitude', 'latitude', color='green', ax=ax, transform =crs, s = 2, alpha=0.2, cmap='viridis', zorder=10)
    dfc_.plot.scatter('longitude', 'latitude', ax=ax, transform =crs, marker ="*", s = 50, c='yellowgreen', zorder=15)
    #for depth in dfc_.depth.unique() : 
    #    dfc_[dfc_.depth == depth].plot.scatter('longitude', 'latitude', ax=ax, transform =crs, marker = depthmarker[depth], s = 50, c='yellowgreen', zorder=15, label =f'depth = {depth}')
    #    print('ok')
    ax.add_feature(cfeature.LAND,)
    ax.set_extent(bbox)
    #ax.legend()
    #cb = plt.colorbar(a)
    #cb.remove()
    
    gl = ax.gridlines(draw_labels=True,)
    gl.xlabels_top = False
    gl.ylabels_right = False
    
    time = pd.to_datetime(dss.time.mean().values).round('min')
    ax.set_title(f'Swath n°{swath}, Cycle n°{cycle}'+'\n'+f'{time.strftime("%m-%d-%Y %H:%M")}')
    #hand, lab = set_legend(handles, labels)
    #ax.legend(hand, lab)
    fig.tight_layout()
    #path = f'/Users/mdemol/ownCloud/PhD/images/med_images/gif_{swath}_{pos}/{swath}_{cycle}.png'
    #fig.savefig(path, dpi=200, bbox_inches='tight')
    #return fig


In [ ]:
cycle=511
swath = 3
plot_one_cycle_one_swath(cycle, swath, dfs, df, dfc, era)

In [ ]:
cycle=511
swath = 16
plot_one_cycle_one_swath(cycle, swath, dfs, df, dfc, era)


In [ ]:
dfs = browse_swot_250().reset_index()
for swath in [3]:
    for cycle in df.where(df.pass_number==swath).dropna().cycle_number.unique().astype(int):
        plot_one_cycle_one_swath(cycle, swath, dfs, df, dfc, era)


In [ ]:
swath = 16
cycle = 10
fig, ax = plot_one_cycle_one_swath(cycle, swath, dfs)

In [ ]:
x = list(depthcolor.keys())
y = [depthcolor[d] for d in x]
lab = ['depth = 0-1m', 'depth = 15m', 'depth = 50m', 'depth = 100m']
fig, ax= plt.subplots()
for i in range(4):
    ax.scatter(x[i], x[i], c=y[i], label=lab[i])
ax.legend()
handles, labels = ax.get_legend_handles_labels()

_______
# All drifters with nearest SWOT image
Pas les mêmes grilles spatiales, interpolation temporelle des images SWOT pas triviale

In [ ]:
def select_data_time(time, dt_image, swath, dfs, df, dfc, era):
    # SWOT
    dfs_ = dfs.where(dfs.pass_number==int(swath)).dropna()
    nearest_cycle = dfs_.iloc[(dfs_['time']-time).abs().argsort()[:1]].file.values[0]
    dss = xr.open_dataset(nearest_cycle)
    dss['eta'] = dss.cvl_mean_dynamic_topography_cnes_cls_22 + dss.cvl_ocean_tide_fes_2022 + dss.duacs_ssha_karin_2_filtered
    dss.eta.attrs={'long_name':'ADT+ocean tide correction', 'unit':'m'}
    dss['adt'] = dss.cvl_mean_dynamic_topography_cnes_cls_22 + dss.cvl_ocean_tide_fes_2022 + dss.duacs_ssha_karin_2_filtered
    dss.eta.attrs={'long_name':'ADT', 'unit':'m'}
    nearest_cycle = dss.cycle_number.values[0]
    
    # DRIFTERS
    dt_traj = pd.Timedelta('12h')
    df_ = df.where((df.pass_number==swath)&(df.datetime>time-dt_traj)&(df.datetime<time)).dropna()
    dt_image = pd.Timedelta(dt_image)
    dfc_ = dfc.where((dfc.pass_number==swath)&(dfc.cycle_number==nearest_cycle)).dropna()
    dfc_ = dfc_.where((dfc_.datetime>time-dt_image)&(dfc_.datetime<time)).dropna()
    
    #ERA5
    era_ = era.sel(time=time, method='nearest').sortby('latitude')
    
    return dss, df_, dfc_, era_, nearest_cycle

def set_legend(handles, labels):
    lab = np.unique(labels)
    ind = [np.argwhere(np.array(labels) == l)[0][0] for l in lab]
    hand = np.take(handles, ind)
    return hand, lab
    #ax.legend(hand, lab)

def plot_time_one_swath(time, dt_image, swath, dfs, df, dfc, era):
    if swath == 3 : 
        #bbox = [2.7, 6.1, 36.6, 43.4]
        #bbox, pos = [3.5, 6.1, 40, 43.4], 'sup'
        bbox, pos = [2.7, 5.1, 36.6, 40], 'inf'
        
        vmax=0.2
        vmin=-0.2
        
    if swath == 16 : 
        #bbox = [0.9, 3.7, 36.5, 41.5]
        bbox, pos = [0.9, 3.7, 37.7, 41.25], ''
        vmax=0.2
        vmin=-0.2
        
    dss, df_, dfc_, era_ , nearest_cycle= select_data_time(time, dt_image, swath, dfs, df, dfc, era)
    
    
    fig = plt.figure(figsize=(10,10))
    fig.patch.set_facecolor('white')#no transparancy in png
    ax = fig.add_subplot(111, projection=ccrs.Orthographic(np.mean(bbox[:2]), np.mean(bbox[2:])))
    
    #SWOT
    #(dss.eta - dss.eta.median()).plot(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax,cmap='RdBu_r', cbar_kwargs={'pad':0.1})#"location": "left"})#remove median
    (dss.adt - dss.adt.median()).assign_attrs({'long_name':'ADT-median of the snapshot', 'unit':'m'}).plot(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax,cmap='RdBu_r', cbar_kwargs={'pad':0.1})#"location": "left"})#remove median
    #dss.eta.plot.contour(x='longitude', y='latitude', transform=crs, ax=ax, vmin=vmin, vmax=vmax, levels=40, colors='k', linewidths=0.2)
    dss['ve'] = dss.duacs_speed_zonal_abs 
    dss['vn'] = dss.duacs_speed_meridional_abs
    dss_c = dss[['ve', 'vn']].isel(num_lines = slice(0,None,15),num_pixels = slice(0,None,15))
    dss_c = dss_c.where(dss_c.latitude>bbox[2]).where(dss_c.latitude<bbox[3])
    P = dss_c.plot.quiver('longitude', 'latitude','ve','vn', color='k', scale = 20, transform=crs, ax=ax, clip_on = False, add_guide = False)
    ax.quiverkey(P, 0.8, 0.89, 1, 'SWOT 1/15\n'+r'1 $m.s^{-1}$', coordinates='figure', labelpos='N')

    
    #ERA5
    era_ = era_.sel(longitude=slice(bbox[0], bbox[1]), latitude=slice(bbox[2], bbox[3]))
    Q = era_.plot.quiver('longitude', 'latitude', 'u10', 'v10', ax=ax, transform =crs, scale=250, color='magenta', clip_on = False, add_guide = False)
    ax.quiverkey(Q, 0.8, 0.95, 10, 'ERA5 wd at 10m\n'+r'10 $m.s^{-1}$', coordinates='figure', labelpos='N')
    
    #drifters

    ax.scatter(df_.longitude, df_.latitude, transform =crs, marker =".", s = 5, c = df_.depthcolor, zorder=105)
    df__ = df__ = df_.set_index('datetime').groupby('drifter_id').last()
    ax.scatter(df__.longitude, df__.latitude, transform =crs, marker ="x", s = 20, c = 'r', zorder=105)
    print(len(dfc_))
    ax.scatter(dfc_.longitude, dfc_.latitude, transform =crs, marker ="*", s = 40, c = 'gold', zorder=105)
    #for depth in dfc_.depth.unique() : 
    #    dfc_[dfc_.depth == depth].plot.scatter('longitude', 'latitude', ax=ax, transform =crs, marker = depthmarker[depth], s = 50, c='yellowgreen', zorder=15, label =f'depth = {depth}')
    #    print('ok')
    ax.add_feature(cfeature.LAND,)
    ax.set_extent(bbox)
    #ax.legend()
    #cb = plt.colorbar(a)
    #cb.remove()
    
    gl = ax.gridlines(draw_labels=True,)
    gl.xlabels_top = False
    gl.ylabels_right = False
    
    swot_time = pd.to_datetime(dss.time.mean().values).round('min')
    ax.set_title(f'Datetime : {time.strftime("%m-%d-%Y %H:%M")}'+ '\n' +f'SWOT : Swath n°{swath}, Cycle n°{nearest_cycle}'+f' - {swot_time.strftime("%m-%d-%Y %H:%M")}')
    #hand, lab = set_legend(handles, labels)
    #ax.legend(handles, labels)
    fig.tight_layout()
    path = f'/Users/mdemol/ownCloud/PhD/images/med_images/gif_time/gif_{swath}_{pos}/{time.strftime("%m%d%Y%H%M")}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    #return fig


In [ ]:
swath = 3
plot_time_one_swath(pd.to_datetime('2023-04-11 20:00:00'), '6h', swath, dfs, df, dfc, era)

In [ ]:
swath = 3
dt_image = '6h'
tmin, tmax = df[df.pass_number == swath].datetime.min().round('d'), df[df.pass_number == swath].datetime.max()
tlist = pd.date_range(tmin, tmax+pd.Timedelta(dt_image), freq=dt_image)
for t in tlist : 
    plot_time_one_swath(t, dt_image, swath, dfs, df, dfc, era)

In [ ]:
tmax

# GIF

In [1]:
# make gif
from PIL import Image


def make_gif(frame_folder, duration=200):
    frames = [Image.open(image) for image in sorted(glob(f"{frame_folder}/*.png"))]
    frame_one = frames[0]
    frame_one.save(
        f"{frame_folder}.gif",
        format="GIF",
        append_images=frames,
        save_all=True,
        duration=duration,
        loop=0,
    )



In [3]:
make_gif('/Users/mdemol/ownCloud/PhD/images/med_images/gif_time/gif_3_sup')

In [4]:
make_gif('/Users/mdemol/ownCloud/PhD/images/med_images/gif_time/gif_3_inf')

In [5]:
make_gif('/Users/mdemol/ownCloud/PhD/images/med_images/gif_time/gif_16_')